# ⚡ Módulo 04: La Revolución del Transformer y la Atención
## Capítulo 1: Mecánica de la Atención: De la Búsqueda en Tablas Hash a Scaled Dot-Product y Multi-Head Attention

> *"Un programador clásico entiende una tabla hash O(1): buscas una Query contra un conjunto de Keys y obtienes el Value exacto. La Atención no es más que una tabla hash probabilística y diferenciable: calculas la similitud de la Query con todas las Keys, aplicas Softmax para obtener una distribución de pesos, y devuelves una suma ponderada continua de los Values. 'Attention Is All You Need' no fue solo un paper: fue el nacimiento de la arquitectura dominante de la civilización computacional contemporánea."*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mcarbonell/algo-to-ai/blob/main/notebooks/04_transformers/01_attention_mechanics.ipynb)

---

### ⚙️ Inicialización del Entorno
Cargamos las librerías matemáticas y fijamos semillas para asegurar reproducibilidad determinista.

In [ ]:
# !pip install -q numpy matplotlib torch
from typing import Tuple, List, Dict, Optional
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)
print("✅ Entorno listo para estudiar la mecánica de la Atención from scratch")

---

## 1. 📜 Contexto Histórico y Proceso de Descubrimiento

### El Cuello de Botella de Sequence-to-Sequence (Sutskever et al., 2014)
Hacia 2014, el modelo dominante para traducir texto era la arquitectura **Seq2Seq (Codificador-Decodificador)** basada en RNNs/LSTMs:
1. Una RNN leía una frase en inglés palabra por palabra: *"The European economic area was established in 1994..."*.
2. Todo el significado de esa frase larga debía comprimirse en el **último vector de estado oculto** $h_T$ (típicamente de 512 o 1024 números).
3. Otra RNN intentaba generar la traducción al francés leyendo exclusivamente desde ese vector.

> **La Analogía de la Pizarra:** Imagina leer un párrafo completo de 50 palabras, borrar la pizarra, resumir todo en una sola palabra clave, y pedirle a un traductor que reproduzca el párrafo entero a partir de esa única palabra. Si la frase superaba las 20 palabras, el modelo olvidaba detalles esenciales.

### Dzmitry Bahdanau, Kyunghyun Cho & Yoshua Bengio (2014): Atención Blanda
En un artículo histórico (*"Neural Machine Translation by Jointly Learning to Align and Translate"*), Bahdanau se preguntó:
*¿Por qué forzar al modelo a comprimir todo en un solo vector? Permitamos que el decodificador conserve todos los estados intermedios $h_1, h_2, \dots, h_T$ y, al generar cada palabra, mire hacia atrás y decida a qué palabras de origen prestar atención.* Nació la **Atención Blanda (*Soft Attention*)**.

### La Revolución de Vaswani et al. (2017): "Attention Is All You Need"
En 2017, un equipo de 8 investigadores de Google Brain y Google Research publicaron un artículo con un título desafiante: *"Attention Is All You Need"*.

Se dieron cuenta de algo fundamental:
* Si la atención permite conectar cualquier palabra con cualquier otra palabra en **$O(1)$ pasos de distancia** (sin importar lo lejos que estén),
* y si la atención se puede calcular para todos los tokens **simultáneamente como una multiplicación matricial masiva (GEMM) en GPU**,
* **entonces la recurrencia (RNN/LSTM) es completamente innecesaria**.

Descartaron las RNNs, descartaron las convoluciones, y construyeron el **Transformer** basándose exclusivamente en el mecanismo de autoatención. Este modelo sentó los cimientos de GPT-4, Claude, LLaMA, Gemini, Whisper y AlphaFold.

---

## 2. 🧠 Intuición Geométrica y Mecánica (Mentalidad de Algoritmista)

### La Analogía de la Tabla Hash Continua (Queries, Keys, Values)
Para un programador, el mecanismo de atención es idéntico a una **búsqueda en una base de datos o tabla hash asociativa**:

```
   Concepto Clásico                   Concepto en Atención Neuronal
   ────────────────                   ─────────────────────────────
   Query (Consulta)              ->   Vector que representa: "¿Qué información busco ahora?"
   Key   (Clave de índice)       ->   Vector que representa: "¿Qué tipo de información ofrezco?"
   Value (Contenido)             ->   Vector que contiene:   "La información semántica real"
```

En una tabla hash clásica, comparas la Query contra cada Key de forma exacta: si coincide, devuelves el Value ($0$ o $1$).
En la Atención Neuronal, la comparación se hace mediante el **producto escalar** ($Q \cdot K^T$):
1. **Puntuación de Afinidad (Score):** $S_{i, j} = Q_i \cdot K_j^T$ mide el alineamiento angular entre la Query $i$ y la Key $j$.
2. **Pesos Normalizados:** Aplicamos $\text{Softmax}$ sobre cada fila para obtener probabilidades que suman $1.0$.
3. **Recuperación Ponderada:** Multiplicamos esos pesos por la matriz $V$ para obtener una combinación lineal suave de los valores.

### La Fórmula Maestra de Scaled Dot-Product Attention:
$$\mathbf{\text{Attention}(Q, K, V) = \text{Softmax}\left( \frac{Q K^T}{\sqrt{d_k}} \right) V}$$

### ¿Por qué el Factor de Escala $\frac{1}{\sqrt{d_k}}$? La Física del Producto Escalar
¿Por qué Vaswani y su equipo dividieron por $\sqrt{d_k}$ en lugar de calcular simplemente $\text{Softmax}(Q K^T)$?

Supongamos que los componentes de $q$ y $k$ son variables aleatorias independientes con media 0 y varianza 1.
El producto escalar es una suma de $d_k$ términos:
$$q \cdot k = \sum_{i=1}^{d_k} q_i k_i$$

La varianza de la suma de variables independientes es la suma de sus varianzas:
$$\text{Var}(q \cdot k) = \sum_{i=1}^{d_k} \text{Var}(q_i k_i) = \sum_{i=1}^{d_k} 1 \cdot 1 = \mathbf{d_k}$$

En modelos reales, $d_k = 64$ o $128$. Por tanto, la desviación estándar es $\sqrt{d_k} \approx 8$ u $11$:
* Sin el factor de escala, los productos escalares toman valores extremos como $+35$ o $-30$.
* Al aplicar $\text{Softmax}(x)$, las probabilidades colapsan a un vector one-hot (un $1.0$ y el resto $0.0$).
* **La derivada de la función Softmax en regiones saturadas es exactamente 0**: ¡el gradiente desaparece por completo y el modelo no puede aprender!
* Al dividir por $\sqrt{d_k}$, la varianza del producto escalar vuelve a ser exactamente **$1.0$**, manteniendo las activaciones en la zona sensible del Softmax donde el gradiente fluye con vigor.

### La Máscara Causal (Causal Masking en Modelos Generativos / GPT)
En un modelo de lenguaje autorregresivo (como GPT), el token en la posición $t$ debe predecir el token $t+1$.
¡Está terminantemente prohibido que el token $t$ mire al futuro ($j > t$)!
Para impedirlo, sumamos una matriz triangular superior llena de $-\infty$ a la matriz de afinidad antes del Softmax:
$$M_{i, j} = \begin{cases} 0 & \text{si } j \le i \\ -\infty & \text{si } j > i \end{cases} \quad \implies \quad e^{-\infty} = \mathbf{0}$$

El Softmax transforma automáticamente los $-\infty$ en probabilidad **$0.0$ exacta**, garantizando la causalidad temporal.

---

## 3. 🛠️ Implementación "From Scratch" (Primeros Principios)

Implementemos `scaled_dot_product_attention` y la clase completa `MultiHeadAttentionFromScratch` en NumPy puro.

In [ ]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Softmax numéricamente estable restando el máximo."""
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(
    Q: np.ndarray,
    K: np.ndarray,
    V: np.ndarray,
    mask: Optional[np.ndarray] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Calcula Scaled Dot-Product Attention: Softmax(Q @ K.T / sqrt(d_k) + mask) @ V
    Q, K, V tienen forma (..., T, d_k)
    """
    d_k = Q.shape[-1]
    
    # 1. Producto escalar escalado: (..., T_q, d_k) @ (..., d_k, T_k) -> (..., T_q, T_k)
    scores = (Q @ np.swapaxes(K, -1, -2)) / np.sqrt(d_k)
    
    # 2. Aplicar máscara si existe (ej. máscara causal autoregresiva)
    if mask is not None:
        # Donde mask sea False, colocamos un valor negativo gigante (-1e9) para que Softmax dé 0
        scores = np.where(mask, scores, -1e9)
        
    # 3. Softmax sobre la última dimensión (claves)
    attention_weights = softmax(scores, axis=-1)
    
    # 4. Multiplicar por Values: (..., T_q, T_k) @ (..., T_k, d_v) -> (..., T_q, d_v)
    output = attention_weights @ V
    return output, attention_weights

print("✅ Función scaled_dot_product_attention compilada exitosamente")

### Construcción de `MultiHeadAttentionFromScratch`

En lugar de calcular una sola atención con dimensión $d_{model}$, dividimos la representación en $h$ cabezas paralelas ($d_k = d_{model} / h$), permitiendo que cada cabeza atienda a diferentes relaciones conceptuales simultáneamente:

In [ ]:
class MultiHeadAttentionFromScratch:
    """
    Multi-Head Attention completo en NumPy puro.
    Procesa todas las cabezas en un único bloque tensorial en paralelo.
    """
    def __init__(self, d_model: int, n_heads: int):
        assert d_model % n_heads == 0, "d_model debe ser divisible entre n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        scale = np.sqrt(2.0 / d_model)
        # Proyecciones lineales para Q, K, V y salida O
        self.W_q = np.random.randn(d_model, d_model) * scale
        self.W_k = np.random.randn(d_model, d_model) * scale
        self.W_v = np.random.randn(d_model, d_model) * scale
        self.W_o = np.random.randn(d_model, d_model) * scale

    def forward(
        self,
        x: np.ndarray,
        is_causal: bool = False
    ) -> Tuple[np.ndarray, np.ndarray]:
        B, T, D = x.shape
        H = self.n_heads
        d_k = self.d_k
        
        # 1. Proyecciones lineales: (B, T, D) @ (D, D) -> (B, T, D)
        q = x @ self.W_q
        k = x @ self.W_k
        v = x @ self.W_v
        
        # 2. Desglosar en H cabezas: (B, T, H, d_k) -> trasponer a (B, H, T, d_k)
        q = q.reshape(B, T, H, d_k).transpose(0, 2, 1, 3)
        k = k.reshape(B, T, H, d_k).transpose(0, 2, 1, 3)
        v = v.reshape(B, T, H, d_k).transpose(0, 2, 1, 3)
        
        # 3. Construir máscara causal triangular inferior si se solicita (estilo GPT)
        mask = None
        if is_causal:
            # Matriz booleana triangular inferior: True donde j <= i
            mask = np.tril(np.ones((T, T), dtype=bool))
            
        # 4. Atención por producto escalar sobre todas las cabezas simultáneamente
        out, attn_weights = scaled_dot_product_attention(q, k, v, mask=mask)
        
        # 5. Recombinar cabezas: (B, H, T, d_k) -> (B, T, H, d_k) -> (B, T, D)
        out = out.transpose(0, 2, 1, 3).reshape(B, T, D)
        
        # 6. Proyección final de salida: (B, T, D) @ (D, D) -> (B, T, D)
        output = out @ self.W_o
        return output, attn_weights

print("✅ MultiHeadAttentionFromScratch compilado exitosamente")

### Experimento Visual: El Mapa de Calor de Atención Causal (*Causal Attention Heatmap*)
Visualicemos la matriz de pesos de atención sobre una secuencia de 8 tokens con máscara causal autorregresiva:

In [ ]:
tokens = ["El", "Transformer", "revolucionó", "la", "inteligencia", "artificial", "moderna", "."]
T = len(tokens)
d_model = 32
n_heads = 4

# Crear embeddings aleatorios para la secuencia (1, T, d_model)
x_seq = np.random.randn(1, T, d_model)

mha = MultiHeadAttentionFromScratch(d_model=d_model, n_heads=n_heads)
_, attn_maps = mha.forward(x_seq, is_causal=True)

# Visualizar los mapas de atención de las 4 cabezas
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for h in range(n_heads):
    ax = axes[h]
    im = ax.imshow(attn_maps[0, h], cmap='viridis', vmin=0.0, vmax=1.0)
    ax.set_title(f"Cabeza de Atención #{h+1}", fontsize=11, fontweight='bold')
    ax.set_xticks(range(T))
    ax.set_yticks(range(T))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(tokens, fontsize=8)
    ax.grid(False)

plt.tight_layout()
plt.show()
print("👁️ Observa la estructura triangular inferior estricta: ningún token puede mirar al futuro.")

---

## 4. ⚡ Transición a PyTorch Moderno

Verifiquemos que nuestra función `scaled_dot_product_attention` from-scratch coincide con la implementación oficial acelerada `F.scaled_dot_product_attention` de PyTorch:

In [ ]:
np.random.seed(42)
B, H, T, d_k = 2, 4, 6, 8
Q_raw = np.random.randn(B, H, T, d_k).astype(np.float32)
K_raw = np.random.randn(B, H, T, d_k).astype(np.float32)
V_raw = np.random.randn(B, H, T, d_k).astype(np.float32)

# 1. Atención From Scratch
out_scratch, _ = scaled_dot_product_attention(Q_raw, K_raw, V_raw)

# 2. Atención en PyTorch Oficial (F.scaled_dot_product_attention)
t_Q = torch.from_numpy(Q_raw)
t_K = torch.from_numpy(K_raw)
t_V = torch.from_numpy(V_raw)
out_torch = F.scaled_dot_product_attention(t_Q, t_K, t_V).numpy()

diff = np.max(np.abs(out_scratch - out_torch))
print(f"Diferencia máxima absoluta con PyTorch: {diff:.2e}")
assert np.allclose(out_scratch, out_torch, atol=1e-5)
print("✅ Coincidencia matemática perfecta: nuestra atención reproduce el kernel oficial de PyTorch")

---

## 5. 🎯 Retos & Experimentos ("Tinker Time")

### Reto 1: Demostración Empírica del Colapso de Gradiente sin $\sqrt{d_k}$
Comprobemos empíricamente qué ocurre cuando aumentamos la dimensión de las claves a $d_k = 128$ y omitimos el escalado:

In [ ]:
d_k_large = 128
T_test = 20
Q_large = np.random.randn(1, T_test, d_k_large)
K_large = np.random.randn(1, T_test, d_k_large)

# Con escalado
scores_scaled = (Q_large @ K_large.transpose(0, 2, 1)) / np.sqrt(d_k_large)
weights_scaled = softmax(scores_scaled, axis=-1)

# Sin escalado
scores_unscaled = Q_large @ K_large.transpose(0, 2, 1)
weights_unscaled = softmax(scores_unscaled, axis=-1)

plt.figure(figsize=(9, 4))
plt.plot(weights_scaled[0, 0], 'g-o', label='Con Escalado 1/sqrt(d_k) (Distribución saludable y suave)', linewidth=2)
plt.plot(weights_unscaled[0, 0], 'r--s', label='Sin Escalado (Colapso one-hot / Gradiente Cero)', linewidth=1.5)
plt.title('El Efecto del Factor de Escala en la Distribución de Atención (d_k = 128)')
plt.xlabel('Posición de la Clave (Key index)')
plt.ylabel('Probabilidad de Atención')
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend()
plt.show()

print(f"Máxima probabilidad sin escalado: {np.max(weights_unscaled):.4f} (El 99.9% de los pesos concentrados en 1 token)")
print(f"Máxima probabilidad con escalado: {np.max(weights_scaled):.4f} (Distribución balanceada y diferenciable)")

### Reto 2 (Para resolver): Implementar Cross-Attention (Atención Cruzada)
En modelos como el Transformer original, Whisper o Stable Diffusion, las consultas $Q$ provienen de una secuencia (ej. el texto generado o el decodificador), mientras que las claves $K$ y valores $V$ provienen de otra secuencia distinta (ej. el audio o el texto condicionante):

Implementa a continuación una función `cross_attention(q_seq, kv_seq)`:

In [ ]:
# TU CÓDIGO DEL RETO 2 AQUÍ
def cross_attention(q_seq: np.ndarray, kv_seq: np.ndarray) -> np.ndarray:
    """
    Calcula la atención cruzada donde Q tiene longitud T_q y K, V tienen longitud T_kv.
    """
    # Tu implementación aquí
    pass

---

## 6. 📚 Referencias Fundamentales & Lecturas Recomendadas

### 📄 Papers Seminales
1. **Vaswani, A., et al. (2017):** *"Attention Is All You Need"*, Advances in Neural Information Processing Systems (NeurIPS 2017). [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
   * *¿Qué leer?* Sección 3.2 ("Attention"): la definición matemática fundamental de Scaled Dot-Product y Multi-Head Attention.
2. **Bahdanau, D., Cho, K., & Bengio, Y. (2014):** *"Neural Machine Translation by Jointly Learning to Align and Translate"*, ICLR 2015. [arXiv:1409.0473](https://arxiv.org/abs/1409.0473)
   * *¿Qué leer?* La introducción de la atención como mecanismo de alineamiento suave.
3. **Dao, T., et al. (2022):** *"FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness"*, NeurIPS 2022. [arXiv:2205.14135](https://arxiv.org/abs/2205.14135)
   * *¿Qué leer?* Cómo calcular la atención exacta mediante *tiling* sin materializar la gigantesca matriz $T \times T$ en memoria HBM de la GPU.

### 🔗 Lecturas Visuales Recomendadas
* **Jay Alammar:** [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) - La explicación visual de referencia que ayudó a toda una generación de ingenieros a comprender los Transformers.